---
title: ""
lang: uk
format:
  typst:
    mainfont: "Times New Roman"
    fontsize: 14pt
    margin: { x: 2.5cm, y: 2.5cm }
  docx: default
  pdf:
    pdf-engine: lualatex
    mainfont: "Times New Roman"
---

```{=typst}
// Configure the page.
#set page(
  paper: "a4",
  // The margins depend on the paper size.
  margin: (x: 2cm, y: 1.5cm),
  numbering:  (current, total) => if current > 1 { current }, number-align: right, 
)

// Configure styling
#set par(
  leading: 0.8em,
  spacing: 1em,
  justify: true,
)
#show table: set text(size: 9pt)

#align(top + center)[
  Міністерство освіти і науки України \
  Національний університет «Львівська політехніка» \
  Кафедра систем штучного інтелекту \
  \
  #image("logo.png", width: 6cm)
  \
  #set par(spacing: 0.4em)
  #text(size: 24pt, weight: "bold")[Звіт]
  #set par(spacing: 0.8em)

  *про виконання лабораторної роботи № 5* \
  _з дисципліни:_ «Обробка мови методами штучного інтелекту»
]

#align(bottom + right)[
  Виконав: \
  студент групи ШІ-42 \
  Горишевський Олексій Олександрович
]

#pagebreak()
```

# Генерація тексту

У цій роботі ми використаємо мовну модель для генерації тексту.

У класичної мовної моделі є два взаємопов'язані визначення:

1. Оцінити ймовірність вхідного тексту.
2. Видати ймовірнісний розподіл наступного слова для даного префіксу.

Для генерації тексту нам ідеально підходить друге визначення.

## Початок роботи

Будь ласка, заповніть поля `EMAIL`, `NAME` та `GROUP` нижче:

In [1]:
%pip install --quiet --ignore-installed https://static.lp-nlp.com/pypi/lpnlp-2026.10.1-py3-none-any.whl

Note: you may need to restart the kernel to use updated packages.


In [ ]:
import lpnlp

EMAIL = "oleksii.ivanov.shi.2023@lpnu.ua"
lab = lpnlp.start(email=EMAIL, lab="text_generation")

Удачі!


### Завантаження моделі

Тренування мовної моделі з нуля займає багато часу: від кількох годин або днів для маленьких та середніх моделей й аж до кількох місяців чи навіть років для великих. Звичайно, за рахунок розпаралелювання тренування на багатьох GPU, реальний час рідко буває більше місяця-двох.

Для цієї роботи (як і в реальному житті), ми візьмемо претреновану модель. Хтось натренував її за нас. В данному випадку, візьмемо GPT-2 від компанії OpenAI.

Ця модель, як і безліч інших, зберігається на [HuggingFace Models](https://huggingface.co/models). Завантажити та працювати з нею зручно через бібліотеку [HuggingFace Transformers](https://github.com/huggingface/transformers/)

In [3]:
%pip install --quiet transformers

Note: you may need to restart the kernel to use updated packages.


In [4]:
import transformers
import torch

In [5]:
model = transformers.AutoModelForCausalLM.from_pretrained("gpt2")

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

## Токенізація

Кожна модель має словник токенів, з яким вона тренувалася, та правила токенізації (розбиття тексту на токени). Обов'язково слід використовувати той самий словник та метод.

Бібліотека transformers вміє робити це для кожної підтримуваної моделі.

In [6]:
tokenizer = transformers.AutoTokenizer.from_pretrained("gpt2")

Перевіримо токенізатор:

In [7]:
input_text = "Hello, LP NLP!"
token_ids = tokenizer.encode(input_text)
token_ids

[15496, 11, 18470, 399, 19930, 0]

In [8]:
tokenizer.convert_ids_to_tokens(token_ids)

['Hello', ',', 'ĠLP', 'ĠN', 'LP', '!']

Рідкісні слова будуть розбиті на підслова:

In [9]:
tokenizer.convert_ids_to_tokens(tokenizer.encode("My name is Oleksiy Horyshevskyi."))

['My',
 'Ġname',
 'Ġis',
 'ĠOle',
 'ks',
 'iy',
 'ĠH',
 'ory',
 'she',
 'vs',
 'ky',
 'i',
 '.']

## Перевірка моделі -- один крок ітерації

Ми генеруватимемо текст в циклі токен за токеном, зліва направо. Але для початку розберімо, як виглядає один крок такого циклу.


На кожному кроці на вхід моделі подаємо ті токени, які вже згенеровано.

На першому кроці у нас ще нічого не згенеровано. Тому починаємо зі токену `<BOS>` ("begin of sentence"). В різних моделях він виглядає по-різному. Подивимося, що в GPT-2:


In [10]:
tokenizer.bos_token

'<|endoftext|>'

In [11]:
tokenizer.bos_token_id

50256

In [12]:
# Формуємо вхідний батч. Майже завжди для більшої ефективності
# нейронні мережі очікують на вхід кілька незалежних речень
# (або зображень у випадку з комп'ютерним зором)
#
# В нашій роботі ми завжди працюємо лише з одним реченням,
# тож розмір батча дорівнює одинці. Але все одно маємо
# оформити вхід як матрицю:
input_ = torch.LongTensor([[tokenizer.bos_token_id]])
input_

tensor([[50256]])

In [13]:
# Нарешті робимо крок генерації
output = model(input_, return_dict=True)

# На виході модель повертає вектор з logits -- ненормалізованими
# ймовірностями кожного токена в словнику
output.logits.shape

torch.Size([1, 1, 50257])

Чому маємо саме таку розмірність?

* Перша одиниця -- розмір батча, тобто кількість вхідних речень.
* Друга одиниця -- це довжина вхідної послідовності. На першому кроці ми подали лише один токен (bos_token)
* 50257 -- це розмір словника

In [14]:
tokenizer.vocab_size

50257

Поки що модель повернула не ймовірності, а просто якісь числа. Їм треба нормалізувати функцією softmax:

In [15]:
probs = torch.softmax(output.logits, dim=-1)

In [16]:
# Розмірність має збігатися з розміром словника
probs.shape

torch.Size([1, 1, 50257])

In [17]:
# Сума ймовірностей має дорівнювати 1.0
probs.sum()

tensor(1.0000, grad_fn=<SumBackward0>)

Ймовірності можуть ставати дуже маленькими та причиняти проблеми у зв'язку з обмеженою точністю float чисел. Тому прийнято працювати з логарифмами ймовірностей.

In [18]:
log_probs = torch.log_softmax(output.logits, dim=-1)

# Щоб перейти до звичайних ймовірностей, маємо зробити експоненціювання
log_probs.exp().sum()

tensor(1.0000, grad_fn=<SumBackward0>)

Кожному слову в словнику відповідає своя ймовірність бути побаченим
після заданого префікса. Префіксом у нас поки що був лише одни `bos_token`.


In [19]:
# Яка ймовірність, що речення почнеться зі слів "I", "red", "Why"?
for word in ("I", "red", "Why"):
    index = tokenizer.convert_tokens_to_ids(word)
    prob = log_probs[0, 0, index].exp()
    print(f"P({word}) = {prob}")

P(I) = 0.01832083985209465
P(red) = 0.00010652336641214788
P(Why) = 0.0008686729124747217


Тепер ми маємо ймовірностний розподіл по словнику. Можемо обрати слово, яке вважатимемо згенерованим. Тут можливі кілька стратегій, які ми розглянемо в наступних розділах.

## Greedy decoding

Найпростіший (але й не дуже цікавий) спосіб -- це завжди обирати токен з найбільшою ймовірністю:

In [20]:
next_token_id = log_probs.argmax()
next_token_id

tensor(198)

In [21]:
probs[-1, -1, 198]

tensor(0.0623, grad_fn=<SelectBackward0>)

In [22]:
tokenizer.convert_ids_to_tokens([next_token_id])  # Наш перший згенерований токен

['Ċ']

Зберемо код докупи та додамо цикл. В циклі ми продовжуватимемо генерувати текст токен за токеном, поки не настане одна з двох умов:
1. Модель видала спецальний токен `eos_token` (end of sentence)
2. Довжина згенерованого тексту перевищила певний поріг `max_len`

У хорошої моделі в більшості випадків має спрацьовувати перша умова зупинки. Проте іноді модель може впасти в безкінчений цикл. Щоб цьому запобігти, маємо другу умову.

In [23]:
@torch.no_grad()
def greedy_decode(model, tokenizer, max_len=50):
    start_index = tokenizer.bos_token_id
    result = [start_index]

    while len(result) < max_len:

        # Передбачення ймовірностней наступного токена
        input_ = torch.LongTensor([result])
        output = model(input_, return_dict=True)
        probs = torch.softmax(output.logits, dim=-1)

        # Обираємо токен, що має найбільшу ймовірність
        token_index = probs[0, -1].argmax()

        # Зупиняємося, якщо досягли кінця
        if token_index == tokenizer.eos_token_id:
            break

        # Додаємо обраний токен в згенерований текст
        result.append(token_index.item())

    return result


generated_token_ids = greedy_decode(model, tokenizer)

Маємо список згенерованих індексів токенів, який починається ось так:

In [24]:
generated_token_ids[:10]

[50256, 198, 464, 717, 640, 314, 2497, 262, 649, 2196]

Перетворимо їх в текст:

In [25]:
tokens = tokenizer.convert_ids_to_tokens(generated_token_ids)
tokens[:10]

['<|endoftext|>',
 'Ċ',
 'The',
 'Ġfirst',
 'Ġtime',
 'ĠI',
 'Ġsaw',
 'Ġthe',
 'Ġnew',
 'Ġversion']

In [26]:
generated_text = "".join(tokens)
generated_text

'<|endoftext|>ĊTheĠfirstĠtimeĠIĠsawĠtheĠnewĠversionĠofĠtheĠgame,ĠIĠwasĠsoĠexcited.ĠIĠwasĠsoĠexcitedĠtoĠseeĠtheĠnewĠversionĠofĠtheĠgame,ĠIĠwasĠsoĠexcitedĠtoĠseeĠtheĠnewĠversionĠofĠtheĠgame,ĠIĠwasĠsoĠexcitedĠto'

In [27]:
lab.checkpoint("greedy decode", generated_text)

Відповідь правильна ✅



'<|endoftext|>ĊTheĠfirstĠtimeĠIĠsawĠtheĠnewĠversionĠofĠtheĠgame,ĠIĠwasĠsoĠexcited.ĠIĠwasĠsoĠexcitedĠtoĠseeĠtheĠnewĠversionĠofĠtheĠgame,ĠIĠwasĠsoĠexcitedĠtoĠseeĠtheĠnewĠversionĠofĠtheĠgame,ĠIĠwasĠsoĠexcitedĠto'

### Примітка: Byte-pair encoding (BPE)

Наша модель використовує subword токенізацію, а саме byte-pair encoding (BPE). В сучасному NLP це найрозповсюдженіший спосіб токенізації. Детально можете подивитися в [цьому відео](https://www.youtube.com/watch?v=tOMjTCO0htA).

Для наших цілей зараз важливо, що BPE заміняє пробіли на спеціальні Unicode-символи "Ġ". Серед інших моделей широко поширений варіант "▁" (зверніть увагу, це не звичайний символ підкреслення "_"). Щоб отримати чистий текст, треба виконати наступну заміну:

In [28]:
def bpe_decode(s):
    result = s.replace("Ġ", " ")
    result = result.replace("Ċ", "\n")
    return result

bpe_decode(generated_text)

'<|endoftext|>\nThe first time I saw the new version of the game, I was so excited. I was so excited to see the new version of the game, I was so excited to see the new version of the game, I was so excited to'

Але краще довіритися токенізатору й зробити цю роботу за нас:

In [29]:
tokenizer.convert_tokens_to_string(tokens)

'<|endoftext|>\nThe first time I saw the new version of the game, I was so excited. I was so excited to see the new version of the game, I was so excited to see the new version of the game, I was so excited to'

## Generic decoding function

Обирати слово з найбільшою ймовірністю -- не найкращий варіант для генерації тексту хоча б тому, що він завжди детерміновано призводить до однієї послідовності. Нижче ми подивимося на цікавіші альтернативи.

Цикл генерації залишиться той самий, що і в `greedy_decode()`. Відрізнятися буде лише один рядок -- той, в якому ми приймали рішення, яке слово обрати. Для зручності, винесемо цей рядок в окрему функцію. Ця функція прийматиме на вхід ймовірностний розподіл по словнику і повертає обраний токен.

In [30]:
def greedy_choice(probs):
    return probs.argmax()

Функцію генерації також трохи переробимо.

По-перше, додамо параметр `sample_fn` -- це має бути функція, яка обирає слово з ймовірностного розподілу, наприклад, `greedy_choice`.

По-друге, для зручності виконуватимемо BPE декодинг у середині функції генерації.

In [31]:
@torch.no_grad()
def generate(model, tokenizer, sample_fn, max_len=50, bpe_decode=True):
    start_index = tokenizer.bos_token_id
    result = [start_index]

    while len(result) < max_len:

        # Передбачення ймовірностней наступного токена
        input_ = torch.LongTensor([result])
        output = model(input_, return_dict=True)
        probs = torch.softmax(output.logits[0, -1], dim=-1)

        # Обираємо токен, що має найбільшу ймовірність
        token_index = sample_fn(probs)     # <---------------- цей рядок змінено

        # Зупиняємося, якщо досягнули кінця
        if token_index.item() == tokenizer.eos_token_id:
            break

        # Додаємо обраний токен в згенерований текст
        result.append(token_index.item())

    if bpe_decode:
        tokens = tokenizer.convert_ids_to_tokens(result)
        result = tokenizer.convert_tokens_to_string(tokens)

    return result


generate(model, tokenizer, greedy_choice)

'<|endoftext|>\nThe first time I saw the new version of the game, I was so excited. I was so excited to see the new version of the game, I was so excited to see the new version of the game, I was so excited to'

## Simple sampling

Перший альтернатива -- це sampling. Тепер ми обираємо наступний токен випадково, але пропорційно до ймовірностей.

In [32]:
def simple_sample(probs):
    return torch.multinomial(probs, num_samples=1)[0]

# Згенеруємо 5 речень
for i in range(1, 6):
    result = generate(model, tokenizer, simple_sample)
    print(f"#{i}: {result}")
    print()

#1: <|endoftext|>The G5 fenders with logos MotoGP installed have always been one of the original design philosophies in the sport. With the formation of the MotoGP front wing in the 1993-94 season, a bit of a milestone when it came to being

#2: <|endoftext|>Watch star Pettigrew call the Green Bay Packers play-by-play reports he divulged in the late night newsdesk.

Carson Palmer was good. All he had to do was continue listening to PFT's Mike O

#3: <|endoftext|>Also, I noticed that the Professionals for Justice was trending down, but it's no longer trending, which is obviously still better.

16 15 11/19/2016 7:06:52 Bishop Dwellings Ghost Research Over 100 In

#4: <|endoftext|>Actress Leigh Gilligan shared her advice to her sister Claire Bellorell:

"I am very much sorry that Marc Aniston has not resigned - I want to be sure that this is all good for all my job prospects and peace

#5: <|endoftext|>(Last week's Advance Media's "Don't Miss Your 100 Best Destination":

Mark Twain's Grea

## Sampling with temperature

Ми також можемо впливати на генерацію параметром температури softmax.

Більші значення температури призводять до того, що різниця між ймовірностями токенів зменшується, тобто розподіл стає більш рівномірним. На практиці це означає, що менш ймовірні варіанти обиратимуться частіше і згенерований текст може бути цікавішим. Однак якщо продовжувати піднімати температуру, то текст спочатку втратить зв'язність, далі почнуть розпадатися слова та граматичність.

Менші значення температури змінюють розподіл таким чином, що основна ймовірніста маса припадає на невелику кількість топових токенів. При температурі 0 вся ймовірність дістанеться одному токену й семплінг перетвориться на greedy decoding.

Згенеруємо тексти з різною температурою:

In [33]:
for temperature in (0.1, 0.3, 0.5, 0.8, 1.0, 1.25, 1.5, 2.0, 3.0, 5.0):

    def sample_with_temp(probs):
        updated_probs = probs ** (1 / temperature)
        updated_probs /= updated_probs.sum()
        return simple_sample(updated_probs)

    print(f"Sampling with temperature={temperature}")
    result = generate(model, tokenizer, sample_with_temp, max_len=15)
    print(result)
    print()

Sampling with temperature=0.1
<|endoftext|>
The first time I saw the new version of the game, I

Sampling with temperature=0.3
<|endoftext|>
The first thing you'll notice when you open the app is that

Sampling with temperature=0.5
<|endoftext|>
The Christian Science Monitor has published a new study that shows that the

Sampling with temperature=0.8
<|endoftext|>Contact - Esther Bloomer

The state Department of Public Health believes

Sampling with temperature=1.0
<|endoftext|>[ XIV ] [ XV ] ++/. [ XVI ] ++/#X

Sampling with temperature=1.25
<|endoftext|>Eleven volumes, 50 000 CDDs are gathered availble by Monster

Sampling with temperature=1.5
<|endoftext|>Figvey went through 180 filled pillow cap farms, some soft pot

Sampling with temperature=2.0
<|endoftext|>amiX simpletower Mü snapITalk MP header IvyMedia spring will

Sampling with temperature=3.0
<|endoftext|>blers develop silent Del Thymis uncovered Appalo going evolution sand p267

Sampling with temperature=5.0
<|endoftext|>IT

In [34]:
# Яке значення `temperature` здається вам оптимальною?
lab.checkpoint("softmax temperature", 0.8)

Відповідь правильна ✅
Окей, добре


0.8

## Top-k sampling

In [35]:
def top_k_sampling(probs, k):
    topk = probs.topk(k)
    index = torch.multinomial(topk.values, num_samples=1)[0]
    # print(f"Top {k} words take {topk.values.sum():%} probability mass")
    return topk.indices[index]

In [36]:
k = 15
sample_fn = lambda probs: top_k_sampling(probs, k=k)
for i in range(1, 6):
    result = generate(model, tokenizer, sample_fn, max_len=20)
    print(f"#{i}: {result}")
    print()

#1: <|endoftext|>AUSTIN - The first thing I noticed when I read about the Texas Rangers' decision to

#2: <|endoftext|>A woman walks down the street in San Francisco on Sunday, Jan. 21, 2017.


#3: <|endoftext|>"He's a good friend," said one woman, who asked not to be identified and declined

#4: <|endoftext|>The United Kingdom and the Netherlands have been locked in a civil war over whether to allow transgender people

#5: <|endoftext|>
When your car is at your disposal when it gets out of range, a safety check is



## Nucleus (top-p) sampling

In [37]:
def nucleus_sampling(probs, max_p):
    sorted_probs = probs.sort(descending=True)
    cum_prob = 0.0
    sample_indices = []
    sample_probs = []
    for i in range(0, len(sorted_probs.values)):
        p = sorted_probs.values[i]
        cum_prob += p
        sample_probs.append(p)
        sample_indices.append(sorted_probs.indices[i])
        if cum_prob >= max_p:
            break

    index = torch.multinomial(torch.tensor(sample_probs), num_samples=1)[0]
    return sample_indices[index]

In [38]:
for max_p in (0.0, 0.1, 0.3, 0.5, 0.6, 0.8, 1.0):
    sample_fn = lambda probs: nucleus_sampling(probs, max_p=max_p)
    result = generate(model, tokenizer, sample_fn, max_len=20)
    print(f"#{max_p}: {result}")
    print()

#0.0: <|endoftext|>
The first time I saw the new version of the game, I was so excited. I

#0.1: <|endoftext|>"I'm not going to be a big fan of the idea of the NFL being a place

#0.3: <|endoftext|>
"We are very pleased to announce that we have acquired a very successful and highly-respected

#0.5: <|endoftext|>I've been thinking about the next time I'm going to write a blog post about a story

#0.6: <|endoftext|>
Showing results of a test for methamphetamine use.

Results of a test for

#0.8: <|endoftext|>It was the biggest game of the season for Jones, who was third in the NHL with 36

#1.0: <|endoftext|> did that mean Flying Heevie was our wiki-annotation but is Sorry more than me



## Start from prompt

До цього моменту ми генерували текст з нуля. Однак значно кориснішим є задача генерації тексту від певного префікса або "підказки" -- в англійській мові це називається "prompt".

Prompt дозволить нам контролювати тематику згенерованого тексту або, як ми побачимо на іншій лекції, допоможе моделі виконувати різноманітні завдання.

In [39]:
@torch.no_grad()
def generate(model, tokenizer, sample_fn, prompt, max_len=50, bpe_decode=True):

    result = tokenizer.encode(prompt)     # <---- цей рядок додано

    while len(result) < max_len:
        # Передбачення ймовірностней наступного токена
        input_ = torch.LongTensor([result])
        output = model(input_)
        log_probs = torch.log_softmax(output.logits[0, -1], dim=-1)

        # Обираємо токен, що має найбільшу ймовірність
        token_index = sample_fn(log_probs.exp())

        # Зупиняємося, якщо досягнули кінця
        if token_index == tokenizer.eos_token_id:
            break

        # Додаємо обраний токен в згенерований текст
        result.append(token_index.item())

    if bpe_decode:
        tokens = tokenizer.convert_ids_to_tokens(result)
        result = tokenizer.convert_tokens_to_string(tokens)

    return result

In [40]:
k = 3
sample_fn = lambda probs: top_k_sampling(probs, k=k)

# Спробуйте змінити prompt на щось інше

def generate_with_prompt(prompt):
    for i in range(1, 5):
        result = generate(model, tokenizer, sample_fn, prompt=prompt)
        print(f"#{i}: {result}")
        print()


prompts = [
    "One of my favorite things is",
    "The most important thing in life is",
    "I think that the best way to learn is",
    "My favorite programming language is",
    "I like to spend my free time",
]

for prompt in prompts:
    print(f"Prompt: {prompt}")
    generate_with_prompt(prompt)

Prompt: One of my favorite things is
#1: One of my favorite things is the fact that the game is so much fun. It is so easy to learn. I have never played a game that was as easy to learn as this one. I have played it for hours and hours, and I

#2: One of my favorite things is the way the people who are the best at what they do are able to do it without being judged by others.

"I'm not a big believer that you should be judged on your work. I think you

#3: One of my favorite things is the fact that I can't even see the back of the head. I've never had a head before. It's just like when you see a car. It's like a piece of glass. I can see the

#4: One of my favorite things is when I see the kids play in front of my house. They're playing in the backyard, they've played in the backyard. It's a great way to see the kids and see what's going on around them."

Prompt: The most important thing in life is
#1: The most important thing in life is to have a plan. You can't have a plan without a 

# Zero-shot класифікація

Ми будемо досліджувати цю тему на окремій лекції. Але спробуємо застосувати мовну модель для задачі класифікації вже зараз. Хоча наша модель надто маленька й слабка для серйозного використання в zero-shot.


In [41]:
generate(model, tokenizer, greedy_choice, prompt="This film is awful! Actor play is terrible. The plot is dull. I would rate it as a").splitlines()[0]


'This film is awful! Actor play is terrible. The plot is dull. I would rate it as a B+.'

In [42]:
generate(model, tokenizer, greedy_choice, prompt="This film is amazing! The time flies by when you watch it. Definitely recommend! On the scale of 1 to 5, I would rate it as a").splitlines()[0]


'This film is amazing! The time flies by when you watch it. Definitely recommend! On the scale of 1 to 5, I would rate it as a 5.'

In [43]:
lab.answer("Готово!")

Відповідь правильна ✅
Сподіваюсь, ця лаба була не надто важкою :)


'Готово!'

# Real world

У цій роботі ми імплементували кілька методів декодінгу. Але, звичайно, все вже зроблено за нас.

Подивіться на параметри функції [generate()](https://huggingface.co/docs/transformers/v4.21.1/en/main_classes/text_generation#transformers.generation_utils.GenerationMixin.generate) з бібліотеки transformers. Багато з них мають виглядати знайомими.

Повний приклад використання:

In [44]:
from transformers import AutoTokenizer, AutoModelForCausalLM

model = AutoModelForCausalLM.from_pretrained("gpt2")
tokenizer = AutoTokenizer.from_pretrained("gpt2")

prompt = "I hope that in that practical class on text generation you"

inputs = tokenizer(prompt, return_tensors="pt")
outputs = model.generate(**inputs, do_sample=True, top_p=0.8, max_length=50, temperature=0.8, num_return_sequences=5)

tokenizer.batch_decode(outputs, skip_special_tokens=True)

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

['I hope that in that practical class on text generation you will see that this is a real problem.\n\nThis is a question that I want to address, and I am very grateful to the people who have been helping me with this.\n\n',
 'I hope that in that practical class on text generation you will be able to take your first step into the world of computer science with the help of the Computer Science Association of America (CSA). The CSA has been instrumental in helping me gain my',
 'I hope that in that practical class on text generation you will learn a lot about the history of the Bible, and the way that it was written, and that you will learn the history of the world of the Bible."\n\nThe question of how',
 'I hope that in that practical class on text generation you will learn how to make use of the techniques that are available in the other classes. I have learned that the more you use the techniques you will get a better understanding of the principles and techniques that',
 'I hope that 

# The End


In [45]:
lab.answer("Готово!")

Відповідь правильна ✅
Сподіваюсь, ця лаба була не надто важкою :)


'Готово!'